# Supermarket refrigeration dataset inspection

Run from the project directory with a Python 3 Jupyter kernel containing pandas, numpy and matplotlib. Run all cells to regenerate `sensor_data.md` beside this notebook. The CSVs are read in place. This exploratory notebook reads every recording, including data that could later be held out. Freeze future split rules before modeling and reserve independent data for an unbiased final evaluation where possible.

Sources: [Sun et al.](https://doi.org/10.1038/s41597-021-00927-6) and the [Figshare release](https://doi.org/10.6084/m9.figshare.13232627).

In [ ]:
from pathlib import Path
import csv
import re

import numpy as np
import pandas as pd
from IPython.display import display

data_dir = Path("data")
paths = sorted(data_dir.glob("*.csv"))
if not paths:
    raise FileNotFoundError("Start the notebook in the project directory containing data/.")

scenarios = {
    1: "Freezer door open",
    2: "Ice accumulation on freezer evaporator",
    3: "Freezer expansion valve failure",
    4: "Chilled-case evaporator fan motor failure",
    5: "Gas cooler airflow blockage",
    6: "Chilled-case evaporator airflow blockage",
}
expected_cadence = {2: 3}  # Table 4, seconds. Other runs used 1 s.

air_points = {
    "T-LTcase-Ret": "Freezer evaporator return air",
    "T-LTcase-Sup": "Freezer evaporator supply air",
    "T-MTCase-Ret": "Chilled-case evaporator return air",
    "T-MTCase-Sup": "Chilled-case evaporator supply air",
    "T-GC-Fan2-In": "Gas cooler fan 2 inlet air",
    "T-GC-Fan2-Out": "Gas cooler fan 2 outlet air",
}
differences = {
    "Freezer return − supply": ("T-LTcase-Ret", "T-LTcase-Sup"),
    "Chilled case return − supply": ("T-MTCase-Ret", "T-MTCase-Sup"),
    "Gas cooler outlet − inlet": ("T-GC-Fan2-Out", "T-GC-Fan2-In"),
}

def channel_family(name):
    for pattern, family in [
        (r"^T-", "Temperature"), (r"^P-", "Pressure"),
        (r"^W[-_]", "Power"), (r"^M-", "Mass flow"), (r"^F-", "Flow"),
        (r"^<blank", "Unnamed field"),
    ]:
        if re.match(pattern, name):
            return family
    return "Other / calculated / setpoint"

def table(frame):
    def escape(value):
        return str(value).replace("|", r"\|").replace("\n", " ")
    rows = [list(frame.columns), ["---"] * len(frame.columns)]
    rows.extend(frame.itertuples(index=False, name=None))
    return "\n".join("| " + " | ".join(map(escape, row)) + " |" for row in rows)

# Table 4 pairs Baseline A with both fault 1 and fault 6.
baselines = dict(enumerate("ABCDEA", 1))


In [ ]:
inventory, profiles, traces = [], [], {}
for path in paths:
    with path.open(encoding="utf-8-sig", newline="") as source:
        headers = next(csv.reader(source))
    seen = set()
    names = []
    for i, header in enumerate(headers, 1):
        name = header or f"<blank header at column {i}>"
        if name in seen:
            name = f"{name} [duplicate at column {i}]"
        names.append(name)
        seen.add(name)
    raw = pd.read_csv(path, header=0, names=names, low_memory=False)
    timestamp = raw.pop("Timestamp")
    time = pd.to_datetime(timestamp, format="mixed", errors="coerce")
    intervals = time.diff().dt.total_seconds()
    positive = intervals[intervals.gt(0)]
    match = re.match(r"Fault(\d+)_", path.stem)
    fault = int(match[1]) if match else 0
    cadence = expected_cadence.get(fault, 1)
    inventory.append({
        "Run": path.stem, "Scenario": scenarios.get(fault, "Baseline"),
        "Rows": len(raw), "Channels": len(raw.columns),
        "First timestamp": time.min(), "Last timestamp": time.max(),
        "Span (h)": round((time.max() - time.min()).total_seconds() / 3600, 3),
        "Nominal (s)": cadence,
        "Common positive step (s)": positive.mode().iat[0] if len(positive) else "—",
        "Repeated stamps beyond first": int(time[time.notna()].duplicated().sum()),
        "Backward steps": int(intervals.lt(0).sum()),
        "Steps > nominal": int(intervals.gt(cadence).sum()),
        "Invalid stamps": int(time.isna().sum()),
    })
    numeric = raw.apply(pd.to_numeric, errors="coerce")
    finite = numeric.replace([np.inf, -np.inf], np.nan)
    quantiles = finite.quantile([.01, .5, .99])
    for name in raw:
        values = finite[name]
        observed = int(values.count())
        span = values.max() - values.min() if observed else np.nan
        profiles.append({
            "Run": path.stem, "Channel": name, "Family": channel_family(name),
            "Missing": int(raw[name].isna().sum()),
            "Non-numeric": int((raw[name].notna() & numeric[name].isna()).sum()),
            "Infinite": int(np.isinf(numeric[name]).sum()),
            "Finite": observed, "Constant": bool(observed and span == 0),
            "Min": values.min(), "P01": quantiles.at[.01, name],
            "Median": quantiles.at[.5, name], "P99": quantiles.at[.99, name],
            "Max": values.max(),
        })
    # Only the plot data are reduced. Full-resolution rows feed all summary statistics.
    for label, (a, b) in differences.items():
        if a in finite and b in finite:
            finite[label] = finite[a] - finite[b]
    hour = time.dt.hour + time.dt.minute / 60 + time.dt.second / 3600
    traces[path.stem] = finite.groupby((hour * 12).floordiv(1) / 12).median()
    del raw, numeric, finite

runs = pd.DataFrame(inventory)
stats = pd.DataFrame(profiles)
channels = (stats.groupby(["Family", "Channel"], sort=False)
            .agg(Runs=("Run", "size"), Missing=("Missing", "sum"),
                 Finite=("Finite", "sum"), Constant_runs=("Constant", "sum"))
            .reset_index())
display(runs, channels, stats)


In [ ]:
comparisons = {}
for number, baseline_letter in baselines.items():
    fault = next((run for run in traces if run.startswith(f"Fault{number}_")), None)
    baseline = f"BaselineTest{baseline_letter}"
    if fault and baseline in traces:
        comparisons[number] = (fault, baseline)

observations = []
for number, (fault, baseline) in comparisons.items():
    # Compare coincident five-minute clock bins for each air-side quantity.
    plots = list(air_points) + list(differences)
    offset = traces[fault].reindex(columns=plots) - traces[baseline].reindex(columns=plots)
    for name in plots:
        delta = offset[name].dropna()
        if not delta.empty:
            observations.append({
                "Fault": number, "Quantity": air_points.get(name, name),
                "Paired bins": len(delta), "Median shift (°F)": round(delta.median(), 2),
                "P90 absolute shift (°F)": round(delta.abs().quantile(.9), 2),
            })
shifts = pd.DataFrame(observations)
display(shifts)


In [ ]:
# Compact channel dictionary, using published names where available.
tokens = {
    "LT": "freezer circuit", "MT": "chilled circuit", "LTcase": "freezer case",
    "MTcase": "chilled case", "GC": "gas cooler", "BPHX": "brazed-plate heat exchanger",
    "SUC": "suction", "DIS": "discharge", "LIQ": "liquid line", "srf": "surface",
    "Ret": "return", "Sup": "supply", "EEVout": "electronic expansion valve outlet",
    "EEVin": "electronic expansion valve inlet", "H20": "water", "C02": "CO₂",
    "Fan1": "fan 1", "Fan2": "fan 2", "IN": "inlet", "OUT": "outlet",
    "CONDENSOR": "condenser", "OilSepIn": "oil separator inlet",
    "MTrack": "chilled compressor rack", "FalseLoad": "simulated refrigeration load",
    "LTcooler": "freezer cooler", "MTcooler": "chilled cooler", "CompRack": "compressor rack",
    "FlashTank": "flash tank", "BP": "bypass",
}
tokens = {key.lower(): value for key, value in tokens.items()}
derived = {
    "SupHCompSuc": "Compressor suction superheat", "SupHCompDisc": "Compressor discharge superheat",
    "SupHEvap1": "Evaporator 1 superheat", "SupHEvap2": "Evaporator 2 superheat",
    "SubClComCond": "Condenser subcooling (exact point unverified)",
    "SubcoolCond1": "Condenser subcooling 1", "SubcoolCond2": "Condenser subcooling 2",
    "SuncoolLiq": "Likely liquid-line subcooling",
    "RefHSct": "Likely refrigerant suction enthalpy", "RefHLiq": "Likely liquid refrigerant enthalpy",
    "Tsetpt": "Temperature setpoint", "RHsetpt": "Relative humidity setpoint",
    "AirHRet": "Return-air enthalpy", "AirHSup": "Supply-air enthalpy",
    "TstatSuc": "Likely suction saturation temperature",
    "TstatCondExit": "Likely condenser exit saturation temperature",
    "TstatDisc": "Likely discharge saturation temperature",
    "TstatSubClExit": "Likely subcooler exit saturation temperature",
    "CapaAirside": "Air-side capacity", "CapaRefrside": "Refrigerant-side capacity",
    "EnergyBalance": "Energy balance", "EERA": "Energy efficiency ratio (A variant)",
    "EER": "Energy efficiency ratio",
}

def describe(name):
    original = name.split(" [duplicate", 1)[0]
    if original in air_points:
        return air_points[original]
    if original.startswith("<blank"):
        return "Unnamed CSV field, meaning unknown"
    if original in derived:
        return derived[original]
    if re.fullmatch(r"T-\d+", original):
        return "Numbered temperature probe, placement unverified"
    if "spare" in original.lower():
        return "Spare temperature input, placement unverified"
    _, *parts = re.split("[-_]", original)
    location = " / ".join(tokens.get(part.lower(), re.sub(r"^COMP(\d+)$", r"compressor \1", part)) for part in parts)
    return f"{channel_family(original)} at {location}"

summary = (stats.groupby("Channel", sort=False)
           .agg(Min=("Min", "min"), Typical=("Median", "median"), Max=("Max", "max"),
                Missing=("Missing", "sum"), Finite=("Finite", "sum"),
                Constant_runs=("Constant", "sum"), Runs=("Run", "size"))
           .reset_index())
summary.insert(1, "Meaning", summary["Channel"].map(describe))
summary.insert(2, "Unit", summary["Channel"].map(lambda name: "°F" if name in air_points else "—"))
summary["Missing %"] = 100 * summary.pop("Missing")
counts = stats.assign(Rows=stats["Run"].map(runs.set_index("Run")["Rows"])).groupby("Channel")["Rows"].sum()
summary["Missing %"] = summary["Missing %"] / summary["Channel"].map(counts)
summary["Constant runs"] = summary["Constant_runs"].astype(str) + "/" + summary["Runs"].astype(str)
summary = summary.drop(columns=["Constant_runs", "Runs", "Finite"])

run_guide = runs[["Run", "Scenario", "Rows", "Nominal (s)", "Common positive step (s)"]].copy()
fault_baselines = {fault: baseline.removeprefix("BaselineTest") for fault, baseline in comparisons.values()}
run_guide.insert(2, "Baseline", run_guide["Run"].map(fault_baselines).fillna("—"))
report = [
    "# Supermarket refrigeration data quick guide",
    "Commercial CO₂ test rig with a freezer (LT), a chilled case (MT), and a shared gas cooler. "
    "The 11 CSVs contain five baseline runs and six fault runs. "
    "[Study and instrumentation](https://doi.org/10.1038/s41597-021-00927-6) · "
    "[Raw data and authors’ analysis script](https://doi.org/10.6084/m9.figshare.13232627)",
    "## Runs",
    table(run_guide),
    "**Reading the clock.** Timestamp is the recorded clock with unspecified timezone. "
    "Ice accumulation and expansion valve failure use minute-resolution timestamps with many rows per minute. "
    "Preserve row order. Fault files identify the scenario for the run. Confirm onset and defrost intervals before labeling rows.",
    "## Channel dictionary and basic statistics",
    f"**{runs['Rows'].sum():,} rows across {len(summary)} distinct non-timestamp fields.** "
    "Min and max use all finite values. Typical is the median of run medians. "
    "Missing percentages use rows in files containing the channel. Constant runs have a single finite value "
    "across their observed readings. Duplicate headers retain their column position. "
    "Names describe likely locations inferred from headers where source mappings are unavailable. "
    "A dash in Unit means the raw CSV unit is unverified. Extreme values may include invalid readings.",
    table(summary.round(2).fillna("—")),
    "Open [sensor_data.ipynb](sensor_data.ipynb) for per-run profiles, fault comparisons and plots. "
    "Run all cells to regenerate this guide.",
]
Path("sensor_data.md").write_text("\n\n".join(report) + "\n", encoding="utf-8")


In [ ]:
import matplotlib.pyplot as plt

def plot_comparison(fault_number, channels_to_plot):
    """Plot any subset of recorded channels or air temperature differences."""
    fault, baseline = comparisons[fault_number]
    fig, axes = plt.subplots(len(channels_to_plot), 1, sharex=True, squeeze=False,
                             figsize=(10, 1.75 * len(channels_to_plot) + 1))
    for axis, channel in zip(axes[:, 0], channels_to_plot):
        for run in (baseline, fault):
            trace = traces[run]
            if channel in trace:
                # Reindex so missing clock bins leave gaps in the line.
                clock = np.arange(288) / 12
                axis.plot(clock, trace[channel].reindex(clock), label=run)
        axis.set_title(air_points.get(channel, channel), loc="left", fontsize=9)
        if channel in air_points or channel in differences:
            axis.set_ylabel("°F")
        axis.grid(alpha=.2)
    axes[0, 0].legend(loc="upper right", fontsize=8)
    axes[-1, 0].set_xlabel("Recorded hour of day · five-minute medians")
    fig.suptitle(scenarios[fault_number])
    fig.tight_layout()
    plt.show()
    plt.close(fig)

for fault_number in comparisons:
    plot_comparison(fault_number, [*air_points, *differences])
